# UKRI analysis

In [5]:
import pandas as pd
from discovery_child_development import PROJECT_DIR

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'

In [6]:
# AltairSaver = altair_save_utils.AltairSaver()

In [7]:
import utils
from discovery_child_development.utils import analysis_utils as au
from discovery_child_development.utils import plotting_utils as pu

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import importlib
importlib.reload(utils);
from discovery_child_development.getters import crunchbase

2024-07-05 10:13:23,673 - botocore.credentials - INFO - Found credentials in environment variables.
2024-07-05 10:13:24,881 - datasets - INFO - PyTorch version 2.1.2 available.


/opt/homebrew/Caskroom/miniconda/base/envs/discovery_child_development/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [293]:
remove_ids = [
    "10ab4057-d3dd-f5d6-3695-16c99e0c3408",

]
corrected_ids = {
    "993ea918-6457-4654-9fa1-83b5b0e474b9": "operations, internet",
    "a6d4a1e4-b9d6-4dfc-9d82-5a59804b753b": "health",
    "81ba0855-d918-532a-26dd-ef02de9d82ca": "robotics, ai2, health, infancy, sleep",
    "6e4e5ca3-a86c-ad77-caae-4081dfad00c4": "health, infancy, sleep",
    "af4262cb-c13b-fd0f-af20-15b6d37d5879": "operations, internet",
    "289bddd6-ffc3-7fc5-7266-ba067d980288": "nutrition, infancy",
    "312f3bab-9d75-1d8e-ccad-8f886320748f": 'mobile, arts, parenting2, cognitive, games, literacy',
    "6833e632-b1cd-7808-40dc-d53e13eaa1ed": 'social_services, operations, internet',
    "0208e998-7b79-491a-1d5b-ab38c595d30a": 'health, infancy, sleep, wearables',
    "6eeac0ab-d315-4921-b310-b702dd7f7dab": 'literacy, internet',
    "c5161255-2c90-0a1f-48cf-fbf0bf4f65ed": 'cognitive, literacy, internet',
    "f0b12cfb-4e9e-c7b6-e509-53f9750b823d": 'protection, internet',
    "c1012c82-8309-425f-841a-d25e22efce40": 'parenting2, preschool',
    "0d793132-fe09-a79e-97b2-87d101fa1692": 'numeracy, literacy',
    "37d8c499-1b73-8ef7-e709-5e3b95a2fff6": 'mobile, community, social_media, infancy',
    "b2131a21-96bd-4ba2-b0a8-ed4d76228381": 'parenting2, preschool',
    "c9ec0172-c0b0-3e67-806c-0dab3d5b29b4": 'community, preschool',
    "8240a205-0723-495f-9187-b626c6eaca64": 'health, nutrition, infancy, ai2',
    "91cd0169-6065-6b15-efc4-be251c489b99": 'numeracy, literacy, mobile',
    "a84a0d9a-0e76-4b11-b5dd-2f8ce11e64a8": 'mobile, cognitive, sleep',
    "2a01aa15-4bc1-4090-98b1-411e1dc5e481": 'labour_market, preschool',
    "7ef32c3b-1f24-6748-a3bf-4e5fbec3a22c": 'literacy',
    "83feb6a0-73ec-488d-bb36-c2070ebf6c0b": 'health, internet, send',
    "13ad4c16-41f7-7570-8410-3e53fcc17e5a": 'preschool, internet',
    "7d658f54-6a25-42a8-ae37-7afaf03c0a2b": 'health, parenting2, infancy, ai2, sleep',
    "6d323acb-2994-f1a4-0023-b4e871306216": 'internet, preschool, operations',
    "3901c0a3-8c84-1dd5-96b6-68d913160050": 'mobile, cognitive',
    "fda14860-5b24-621d-d72f-135bb0d4d9a4a": 'cognitive',
    "f250a303-f3b8-4831-8724-a9a01e5379ab": 'health, arts, cognitive, mobile',
    "4af1e6dd-01c2-4659-99ef-ab8fd265ea29": "health, send",
    "f206484e-4011-7a0f-6036-8a8c7744aa18": 'ar_vr',
    "052237b3-6927-098c-33d0-887785079c59": 'nutrition',
    "16a8aa30-cc72-473b-8cbd-58e8f47d6ce7": 'mental_health, health, mobile',
    "e251175b-bf0c-4b6f-856d-103459959f30": 'mobile, games, literacy',
    "e9e9b56e-a1f5-ce88-79a6-dff609ade1bb": 'protection, wearables',
    "a5d01802-e1c2-7e10-6f52-bb8a2dae0682": 'mobile, cognitive',
    # e-commerce platforms
    "bbde881e-83f8-5271-63aa-72607590bde9": 'internet',
    "846b45bb-d9f1-a3a6-1317-9c36690d5ff2": 'infancy, internet',
    "a25250e3-1fd8-45ec-941c-2368fe667285": "internet",
    "2fad8b81-f0a3-02e1-08ce-133de86a087c": "internet",
    "d2fb32b0-5279-5ae9-ef13-dd234489b754": "internet",   
    "68739a8e-eea7-379e-37f6-caba42b34555": "internet",
    "eb437580-9f75-ac35-e29e-75a75820590d": "internet, infancy",
    "498bd46d-f7a4-4b34-a77f-2651d0429ba6": "internet, infancy",
    "8894260c-ba28-4465-9998-6a7e127e67d3": "internet",
    "db6ccd4a-bd23-8f5d-2d72-e5f2940ffaba": "internet",
    "74439150-d3bc-af0e-167b-c3967b3d45a1": "internet",
    "208490ad-c0d6-46f0-12e4-ff66e1c5a47e": "internet",
    "a76e4343-3639-4bc4-a49c-31ee951d42f2": "internet",
    "4881273e-0cb7-d097-3588-39941b94a1d2": "internet",
    "430d521b-342e-6afc-1eaf-a93c19108178": "internet",
    "7b1ecb65-7f78-733e-34cf-434bbfcf58de": "internet",
    "23e934ba-2bef-44df-bac5-75e4d48f8902": "internet",
    "3fd93ec1-3839-4fa5-b473-3414182c220f": "internet",
    "8e7bb838-fae4-4566-8b3a-6ef3770a4afe": "internet",
    "783ee238-d4c6-43b1-320a-fa63d673e45f": "internet, infancy",
    "47ea1262-2bc4-419a-ae61-18bec9b79db5": "internet",
    "22983571-fbb2-545f-72b2-984ced8509fc": "internet",
    "1ec1c2bb-5ef1-a385-2be0-919b3ac99a6a": "internet, sleep",
    "3501c694-0b31-46f4-ac52-f5f12cc6771e": "internet",
    "2054ed53-f2c5-444f-aad1-57ddc64333b8": "internet, games",
    "eee146be-aeec-45f9-860d-42659f8dd49b": "internet",
    "d8d31899-e7d7-9990-5db5-f2aab0151337": "internet",
    "aed244ad-c2aa-4200-bbcf-7047e1c51c1d": "internet",
    "fa409a05-9a09-4fd6-ba45-094c9e32d3db": "internet",
    "cdd0d18e-8f2c-a7c2-c488-9d1acea0730e": "internet",
    "8f320b5c-5baa-468d-9cdc-1052c70a83ff": "internet",
    "f3b1b1b4-1b1b-1b1b-1b1b-1b1b1b1b1b1b": "internet",
    # content on the internet
    "4d6d8ad0-ee3d-9055-9fbd-21184adbdcd2": "parenting2, infancy, cognitive, physical, preschool, operations",
    "e169bff5-56d0-4d34-a35d-34db4f41581f": "parenting2, cognitive, mobile",
    "e980de7d-a07c-1e14-0f3c-032f80b8c99d": "cognitive, internet",
    "e7a95398-cffa-4679-85dc-f5990eb070d8": "cognitive, internet",
    "02a1e9ad-1e9f-1c37-c6ab-c6781518f66a": "cognitive, internet",
    "d2f4f2d0-0e6a-4e1d-9e8b-6e3f3a7f7f6b": "cognitive, internet",
    "9f2f3b73-e465-43b3-b818-5e08366849fd": "cognitive, internet",
    "b60d4d88-ce5e-6ed9-ee3c-c677403d43f5": "cognitive, internet",
    "e4d0b3a4-8d2b-4f0b-9b4b-2e7a3f3d7f7d": "communication, ai2",
    "7ce96082-3b68-4be6-abd5-aecf9ba6b6ca": "cognitive, internet",
    "aa86d716-0577-7340-6d72-10bfaa39809c": "cognitive, internet",
    "2fb68030-c697-4a22-8925-33b3b781d698": "cognitive, internet",
    # double-checking other "non-tech" companies
    "f9e8c5b6-7b8d-4d7d-8e3d-4e6b4b4c4b4b": "cognitive, mobile",
    "c8ac330e-4508-4326-8452-457620667ca8": "send, mobile",
    "7f40e309-6489-422f-98c9-59f3db93af43": "health, parenting2, infancy",
    # double checking Society cases
    "34980ace-ced8-744d-ea9c-00f2ff624ea0": "mobile, ai2, cognitive, games",
    "f6048123-1c96-433d-a9d7-dad5fc232a96": "parenting2, nutrition, infancy",
    "80db3243-628e-bf22-6ac7-637c21922637": "mobile, arts, parenting2, emotional",
}    



In [282]:
# ident = "80db3243-628e-bf22-6ac7-637c21922637"
# print(data_df.query("org_id == @ident").topics.iloc[0])
# print(data_df.query("org_id == @ident").text.iloc[0])


In [283]:
cb_df = crunchbase.get_cb_from_s3('organizations')

In [294]:
# Gateway to Research labelled data
importlib.reload(utils);
data_df = (
    utils.load_crunchbase_data()
    .query("topics != 'arts'")
    .query("org_id not in @remove_ids")
    .assign(
        # ignore ids that are not in the corrected_ids dict
        topics=lambda x: x['org_id'].map(corrected_ids).fillna(x['topics'])
    )
    # remove Byju's mega deals from 2018 onwards
    # .query("~(org_id == '15d119e6-d721-3baf-da4b-880891c0c3fd' and year >= 2018)")
    .query("amount <= 400000")
)

In [295]:
from discovery_child_development import PROJECT_DIR, S3_BUCKET
from nesta_ds_utils.loading_saving import S3
data_df.to_csv(PROJECT_DIR / 'outputs/data/data_crunchbase.csv', index=False)
S3.upload_obj(
    data_df,
    bucket = S3_BUCKET,
    path_to = '2024-07-iss-child-development/outputs/data/data_crunchbase.csv'
)

In [296]:
# Taxonomy dataframe
importlib.reload(utils);
topics_df = utils.load_topic_data(is_crunchbase=True)

In [297]:
# Transform to one id and topic pair per row
importlib.reload(utils)
data_exploded_df = utils.explode_data(data_df, is_crunchbase=True).query("topics != 'arts'")

## Baseline trends

Baseline UKRI trends for funding and project counts 

In [298]:
importlib.reload(utils)
baseline_df = utils.get_baseline_crunchbase()

In [299]:
baseline_df.to_csv(PROJECT_DIR / 'outputs/data/baseline_data_crunchbase.csv', index=False)
S3.upload_obj(
    baseline_df,
    bucket = S3_BUCKET,
    path_to = '2024-07-iss-child-development/outputs/data/baseline_data_crunchbase.csv'
)

In [300]:
baseline_df

,year,counts,amount
0,2013,21249,4.235161e+07
1,2014,28412,6.857297e+07
2,2015,34201,1.044885e+08
3,2016,35601,1.208383e+08
4,2017,37020,1.541349e+08
5,2018,41433,2.166157e+08
6,2019,41130,1.975202e+08
7,2020,41158,2.229402e+08
8,2021,51862,4.485599e+08
9,2022,46549,3.684496e+08


In [16]:
trends_baseline = au.ts_magnitude_growth_(
    ts_df = baseline_df,
    year_start = 2019,
    year_end = 2023  
)
trends_baseline

,magnitude,growth
counts,4.327080e+04,12.111253
amount,2.906603e+08,81.751636


In [17]:
fig = pu.ts_smooth(
    baseline_df.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

### Edtech baseline

In [301]:
cb_categories_df = (
    cb_df
    .dropna(subset=['category_list'])
    .assign(_category_list = lambda x: x['category_list'].str.split(','))
    .explode('_category_list')
)[['id', 'name', '_category_list']]

In [302]:
edtech_categories = ['E-Learning', 'EdTech', 'Education']
edtech_ids = list(set(cb_categories_df.query("_category_list in @edtech_categories").id.to_list()))

In [303]:
baseline_funding_df = (
    pd.read_parquet(utils.INPUTS_DATA_DIR / "crunchbase/funding_rounds_full.parquet")
    .drop_duplicates("funding_round_id")
    .query("org_id in @edtech_ids or funding_round_id in @data_df.id")
    .assign(year=lambda df: df.announced_on.apply(lambda x: int(x[:4])))
    .query("year >= 2013 and year <= 2023")
    .rename(columns={"raised_amount_gbp": "amount"})
    .assign(id=lambda df: df["funding_round_id"])
    .query("investment_type in @utils.EARLY_STAGE_DEALS")
    .query("amount <= 400000")    
)

In [304]:
baseline_funding_df

,funding_round_id,funding_round_name,type,permalink,cb_url,rank,created_at,updated_at,country_code,state_code,...,investor_id,investor_name,investor_type,is_lead_investor,investor_types,investor_url,smart_money_auto,smart_money_manual,smart_money,id
167502,1947b974-e69a-4ec9-96e5-cf6a110a5798,Angel Round - Yingyu Mofang Xiu,funding_round,yingyu-mofang-xiu-angel--1947b974,https://www.crunchbase.com/funding_round/yingy...,528694.0,2020-08-16 17:58:22,2020-08-16 17:58:22,CHN,None,...,c9d8d954-3862-0836-fd23-e51fb1c1254d,Gobi Partners,organization,True,venture_capital,https://www.crunchbase.com/organization/gobi-p...,False,False,False,1947b974-e69a-4ec9-96e5-cf6a110a5798
167572,061b6924-8ff9-83af-7129-c63e73b428bc,Seed Round - Moku,funding_round,moku-seed--061b6924,https://www.crunchbase.com/funding_round/moku-...,141063.0,2013-05-22 12:37:28,2018-02-12 23:38:14,ITA,None,...,894cb258-c882-f3d0-af94-261b51adb78a,H-FARM,organization,None,"accelerator,corporate_venture_capital,incubator",https://www.crunchbase.com/organization/h-farm,True,False,True,061b6924-8ff9-83af-7129-c63e73b428bc
167609,f5124fa5-ba64-9e4d-f4f2-098769c5170e,Series A - Edoki Academy,funding_round,edoki-academy-series-a--f5124fa5,https://www.crunchbase.com/funding_round/edoki...,542084.0,2017-09-20 21:52:28,2018-02-12 23:45:10,FRA,None,...,0cab37ac-f2b5-4581-fa24-169ef0a8ffa8,Elaia,organization,None,venture_capital,https://www.crunchbase.com/organization/elaia-...,False,False,False,f5124fa5-ba64-9e4d-f4f2-098769c5170e
167842,e70f5b31-2f6a-be65-93d2-3f94b4acd974,Angel Round - FRM Study Course,funding_round,frm-study-course-angel--e70f5b31,https://www.crunchbase.com/funding_round/frm-s...,323847.0,2014-09-25 16:08:48,2018-02-12 23:19:14,USA,NY,...,None,None,None,None,None,None,None,False,False,e70f5b31-2f6a-be65-93d2-3f94b4acd974
167908,3fa1f269-f304-4b6e-b914-6664deff2731,Angel Round - Xueleyun,funding_round,xueleyun-angel--3fa1f269,https://www.crunchbase.com/funding_round/xuele...,146440.0,2019-09-07 11:25:52,2019-09-07 11:25:52,CHN,None,...,ad7f75f6-be9f-7e0b-274c-c02ef0020265,ShenZhen GTJA Investment Group,organization,None,private_equity_firm,https://www.crunchbase.com/organization/shenzh...,False,False,False,3fa1f269-f304-4b6e-b914-6664deff2731
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172524,a6353188-b857-43a2-8470-16161c3975b4,Venture Round - Learning Explorer,funding_round,learning-explorer-series-unknown--a6353188,https://www.crunchbase.com/funding_round/learn...,35740.0,2024-01-02 10:45:05,2024-01-02 10:45:05,USA,CA,...,None,None,None,None,None,None,None,False,False,a6353188-b857-43a2-8470-16161c3975b4
1172554,2b5f7dff-405f-43de-afac-b005aada210e,Venture Round - Nurture Life,funding_round,nurture-life-series-unknown--2b5f7dff,https://www.crunchbase.com/funding_round/nurtu...,34143.0,2024-01-11 07:21:18,2024-01-11 07:21:18,USA,IL,...,None,None,None,None,None,None,None,False,False,2b5f7dff-405f-43de-afac-b005aada210e
1172563,a5f5f56f-bbac-454d-90b8-20aa808bcaf4,Seed Round - Digiage,funding_round,digiage-430a-seed--a5f5f56f,https://www.crunchbase.com/funding_round/digia...,10003.0,2024-01-03 07:57:46,2024-01-03 07:57:46,TUR,None,...,6fb8848a-d2ca-405f-b909-30917b15a070,APY Ventures,organization,True,venture_capital,https://www.crunchbase.com/organization/apy-ve...,False,False,False,a5f5f56f-bbac-454d-90b8-20aa808bcaf4
1172693,60fb1ae0-ac77-420d-99e0-b0f1d5b10bf7,Seed Round - Globowl,funding_round,globowl-seed--60fb1ae0,https://www.crunchbase.com/funding_round/globo...,28911.0,2024-02-14 04:54:20,2024-02-16 16:40:04,USA,FL,...,c880b60d-cda2-4761-b041-8cf7474b419a,TechRise Chicago,organization,None,venture_capital,https://www.crunchbase.com/organization/techrise,False,False,False,60fb1ae0-ac77-420d-99e0-b0f1d5b10bf7


In [309]:
ts_edtech_counts = utils.get_timeseries(baseline_funding_df, column='id')
ts_edtech_amounts = utils.get_timeseries(baseline_funding_df, column='amount')

In [311]:
ts_edtech_amounts.to_csv(PROJECT_DIR / 'outputs/data/baseline_data_crunchbase_edtech.csv', index=False)
S3.upload_obj(
    ts_edtech_amounts,
    bucket = S3_BUCKET,
    path_to = '2024-07-iss-child-development/outputs/data/baseline_data_crunchbase_edtech.csv'
)

In [1163]:
mag_growth_edtech_df = au.ts_magnitude_growth_(
    ts_df = ts_edtech_amounts,
    year_start = 2019,
    year_end = 2023  
)
mag_growth_edtech_df

,magnitude,growth
amount,6.571816e+06,34.437652


In [1164]:
mag_growth_edtech_df.magnitude.iloc[0]*5/1e+6

32.8590797947033

In [1165]:
fig = pu.ts_smooth(
    ts_edtech_amounts.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

## Insight 0: Overall trends

Early-years project growth of funding and project counts trends



In [270]:
ts_counts = utils.get_timeseries(data_df, column='id')
ts_amounts = utils.get_timeseries(data_df, column='amount')

In [271]:
ts_counts

,year,counts
0,2013,186
1,2014,224
2,2015,265
3,2016,271
4,2017,259
5,2018,257
6,2019,265
7,2020,234
8,2021,293
9,2022,175


In [272]:
ts_amounts

,year,amount
0,2013,2.434701e+05
1,2014,5.037585e+05
2,2015,1.033026e+06
3,2016,1.505170e+06
4,2017,7.962922e+05
5,2018,9.816925e+05
6,2019,1.244579e+06
7,2020,1.194967e+06
8,2021,3.396024e+06
9,2022,1.046174e+06


In [273]:
au.ts_magnitude_growth_(
    ts_df = ts_counts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
counts,215.6,-25.864277


In [274]:
au.ts_magnitude_growth_(
    ts_df = ts_amounts,
    year_start = 2019,
    year_end = 2023  
)

,magnitude,growth
amount,1.509881e+06,69.057166


In [275]:
1.509881e+06

1509881.0

In [276]:
fig = pu.ts_smooth(
    ts_amounts.assign(Total="Total").assign(amount = lambda df: df.amount/1000),
    ["Total"],
    variable= "amount",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [277]:
fig = pu.ts_smooth(
    ts_counts.assign(Total="Total"),
    ["Total"],
    variable= "counts",
    variable_title = "Total funding (£ millions)",
    category_column = "Total",
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig)

alt.Chart(...)

In [279]:
utils.get_data_distribution(
    data_exploded_df.query("year >= 2019 and year <= 2023"), 
    column='type', values=['id', 'amount'])

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,23,0.021,55364.986316,0.007
1,Child care & preschool,127,0.118,610225.151123,0.081
2,Development & learning,232,0.215,2292446.085317,0.304
3,General,420,0.39,2695685.632986,0.357
4,Health,425,0.394,3026034.566599,0.401
5,Parenting,177,0.164,720381.014435,0.095
6,Society,108,0.1,509445.801672,0.067
7,Technology,513,0.476,4232509.547608,0.561


In [61]:
importlib.reload(utils)
ts_df = (
    utils.get_data_distribution(data_exploded_df, column='type', values=['id', 'amount'], ts=True)
    .query("type != 'General'")
)
utils.get_data_magnitude_growth(data_exploded_df, ids=None, column='type', value='amount')

,magnitude,growth,type,counts
4,605206.913320,157.077055,Health,425
2,458489.217063,130.054457,Development & learning,232
3,539137.126597,116.978615,General,420
5,144076.202887,69.707536,Parenting,177
6,101889.160334,43.812530,Society,108
7,846501.909522,41.195625,Technology,513
1,122045.030225,-17.089442,Child care & preschool,127
0,11072.997263,-43.500533,Biosciences,23


In [62]:
fig = pu.ts_smooth(
    ts_df,
    ts_df['type'].unique(),
    variable= "amount",
    variable_title = "",
    category_column = 'type',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [1266]:
nontech_ids = data_exploded_df.query("id not in @tech_ids").org_id.to_list()
# cb_df.query("id in @nontech_ids").sample(10)[['name', 'short_description']]
data_exploded_df.query("org_id in @nontech_ids").groupby("name").sum().sort_values('amount')

,year,amount
name,,
Personal social emotional,4039,1.049447e+03
Communication and language,4038,1.647422e+03
Child protection,8056,1.905043e+03
Oral health,2015,3.207332e+03
Non-tech assessments,2022,3.910476e+03
Neuroscience,10100,3.937627e+03
Social services,18163,1.022692e+04
RCTs,10084,1.707914e+04
Mathematics,22194,1.845920e+04


In [1301]:
# ids = (
#     data_exploded_df
#     .query("org_id in @nontech_ids")
#     .query("name == 'Parenting'")
#     .org_id.to_list()
# )
# pd.set_option('display.max_colwidth', 200)
# (
#     cb_df.query("id in @ids")[['name', 'short_description', 'total_funding_usd']]
#     .sort_values('total_funding_usd', ascending=False)
#     .iloc[0:20]
# )

## Additional helper utils

In [73]:
deal_order = ["n/a", "£0-5M", "£5-20M", "£20-100M", "£100M+"]

def deal_amount_to_range_coarse(
    amount: float, currency: str = "£", categories: bool = True
) -> str:
    """
    Convert amounts to range in millions
    Args:
        amount: Investment amount (in GBP thousands)
        categories: If True, adding indicative deal categories
        currency: Currency symbol
    """
    amount /= 1e3
    if (amount >= 0.001) and (amount <= 5):
        return f"{currency}0-5M" if not categories else f"{currency}0-5M"
    elif (amount > 5) and (amount <= 20):
        return f"{currency}5-20M" if not categories else f"{currency}5-20M"
    elif (amount > 20) and (amount <= 100):
        return f"{currency}20-100M" if not categories else f"{currency}20-100M"
    elif amount > 100:
        return f"{currency}100M+"
    else:
        return "n/a"

In [74]:
tech_ids = (
    data_exploded_df
    # .query("type == 'Technology'")
    .query("subtype in @tech_subtypes")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

In [75]:
1 - data_df.amount.isna().sum() / len(data_df)

1.0

In [76]:
funding_df_ranges = (
    data_df
    .query("id in @tech_ids")
    .assign(_amount = lambda df: df.amount/1000)
    .assign(deal_type=lambda df: df.amount.apply(deal_amount_to_range_coarse))
    .astype({"deal_type": "category"})
    .assign(deal_type=lambda x: x.deal_type.cat.set_categories(deal_order))
    .drop_duplicates('id')
)

In [77]:
# funding_df_ranges

In [78]:
data = (
    funding_df_ranges.groupby(["year", "deal_type"], as_index=True)
    .agg(
        counts=("id", "count"),
        total_amount=("amount", "sum"),
    )
    .reset_index()
    .query("year >= 2013")
    .query("year <= 2023")
    # convert to millions
    .assign(total_amount=lambda df: df.total_amount / 1000)
)
data_wide_df = (
    data.pivot(index="year", columns="deal_type", values="total_amount")
    .fillna(0)
    .astype(int)
    .reset_index()
)


In [79]:
data.groupby("year").sum()

,counts,total_amount
year,,
2013,108,124.429639
2014,134,343.423021
2015,143,609.399647
2016,147,1209.904105
2017,146,330.011927
2018,139,656.101867
2019,131,851.677676
2020,125,785.950728
2021,136,2021.634749


In [80]:
data_wide_df.merge(data.groupby('year').sum().reset_index(), on='year', how='left')

,year,n/a,£0-5M,£5-20M,£20-100M,£100M+,counts,total_amount
0,2013,0,55,46,22,0,108,124.429639
1,2014,0,80,88,174,0,134,343.423021
2,2015,0,95,89,424,0,143,609.399647
3,2016,0,101,158,307,642,147,1209.904105
4,2017,0,109,179,40,0,146,330.011927
5,2018,0,121,201,218,114,139,656.101867
6,2019,0,109,156,236,348,131,851.677676
7,2020,0,90,123,456,115,125,785.950728
8,2021,0,110,170,564,1175,136,2021.634749
9,2022,0,85,163,21,0,77,269.569469


In [81]:
data_wide_counts_df = (
    data.pivot(index="year", columns="deal_type", values="counts")
    .fillna(0)
    .astype(int)
    .reset_index()
)
data_wide_counts_df

deal_type,year,n/a,£0-5M,£5-20M,£20-100M,£100M+
0,2013,0,102,5,1,0
1,2014,0,120,9,5,0
2,2015,0,126,9,8,0
3,2016,0,121,16,7,3
4,2017,0,126,18,2,0
5,2018,0,114,18,6,1
6,2019,0,104,18,6,3
7,2020,0,103,13,8,1
8,2021,1,100,17,13,5
9,2022,0,62,14,1,0


In [82]:
# funding_df_ranges.query("year == 2021 and deal_type == '£100M+'")

In [83]:
# funding_df_ranges.query("year == 2021 and deal_type == '£20-100M'").sort_values('_amount', ascending=False)

In [84]:
cb_df[cb_df.id == "2d16b038-75ec-4af2-9e63-fc5312b88e70"]

,id,name,type,permalink,cb_url,rank,created_at,updated_at,legal_name,roles,...,phone,facebook_url,linkedin_url,twitter_url,logo_url,alias1,alias2,alias3,primary_role,num_exits


## Insight 1: Technology trends

- Magnitude and growth for technology topic overall
- Distribution of different technologies
- Growth of different technologies in UKRI funding


### Overall technology topic growth

In [243]:
EUROPEAN_COUNTRIES = {
    'IRL': 'Ireland',
    'LUX': 'Luxembourg',
    'CHE': 'Switzerland',
    'GBR': 'United Kingdom',
    'ESP': 'Spain',
    'RUS': 'Russia',
    'DEU': 'Germany',
    'FRA': 'France',
    'FIN': 'Finland',
    'SWE': 'Sweden',
    'NLD': 'Netherlands',
    'BEL': 'Belgium',
    'DNK': 'Denmark',
    'CZE': 'Czech Republic',
    'POL': 'Poland',
    'EST': 'Estonia',
    'AUT': 'Austria',
    'MLT': 'Malta',
    'ITA': 'Italy',
    'ROM': 'Romania',
    'CYP': 'Cyprus',
    'NOR': 'Norway',
    'PRT': 'Portugal',
    'BGR': 'Bulgaria',
    'BLR': 'Belarus',
    'UKR': 'Ukraine',
    'SVN': 'Slovenia',
    'ARM': 'Armenia',
    'HUN': 'Hungary',
    'GRC': 'Greece',
    'ISL': 'Iceland',
    'LVA': 'Latvia',
    'LTU': 'Lithuania',
    'HRV': 'Croatia',
    'MKD': 'North Macedonia',
    'BIH': 'Bosnia and Herzegovina',
    'LIE': 'Liechtenstein',
    'SRB': 'Serbia',
    'ALB': 'Albania',
    'SVK': 'Slovakia',
    'GEO': 'Georgia',
    'MDA': 'Moldova',
    'AND': 'Andorra',
    'MNE': 'Montenegro',
    'SMR': 'San Marino',
    # 'USA': 'United States',
    # 'CAN': 'Canada',
}

In [244]:
len(EUROPEAN_COUNTRIES)

45

In [286]:
tech_subtypes = set(topics_df.query("type == 'Technology'").subtype.unique())
tech_subtypes

{'AI', 'Immersive tech', 'Internet', 'Mobile', 'Operations'}

In [287]:
tech_type_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'type'])
    # check only small and medium deals
    .drop_duplicates('id')
    # .query("amount < 100000")
    # check only european deals
    # .query("country_code in @EUROPEAN_COUNTRIES")
)

In [288]:
tech_ids = tech_type_df.id.unique()

In [289]:
ts_amounts_tech = utils.get_timeseries(tech_type_df, column='amount')
ts_counts_tech = utils.get_timeseries(tech_type_df, column='id')
utils.plot_quick_ts(ts_amounts_tech, 'amount')

alt.Chart(...)

In [290]:
mag_growth_df = au.ts_magnitude_growth_(ts_amounts_tech, year_start = 2019, year_end = 2023)
mag_growth_df

,magnitude,growth
amount,846501.909522,41.195625


In [291]:
mag_growth_df.magnitude.iloc[0]*5/1e+6

4.232509547607872

In [292]:
# compare 

### Distribution of different technologies

In [238]:
tech_subtype_df = (
    data_exploded_df
    # All items that belong to the allowed technology subtypes
    .query('subtype in @tech_subtypes')
    # .query("type == 'Technology'")
    # Drop all duplicates that belong to technology type
    .drop_duplicates(['id', 'subtype'])
)

In [239]:
# Total tech funding
amount_total = tech_subtype_df.drop_duplicates('id').query("year >= 2019").amount.sum()

In [240]:
tech_subtype_dist = (
    tech_subtype_df
    .query("year >= 2019")
    .groupby('subtype')
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
    .assign(amount_prop = lambda df: round(df.amount / amount_total, 3))
)

tech_subtype_dist

,subtype,counts,amount,amount_prop
0,AI,173,6.250164e+05,0.148
1,Immersive tech,62,2.846615e+05,0.067
2,Internet,110,2.892629e+06,0.683
3,Mobile,234,1.377192e+06,0.325
4,Operations,89,4.358776e+05,0.103


### Growth of technology topics

In [241]:
column = 'subtype'
value = 'amount'

tech_subtype_ts = (
    tech_subtype_df
    .drop_duplicates(['id', column])
    .groupby(['subtype', 'year'])
    .agg(
        counts=('id', 'nunique'), 
        amount=('amount', 'sum')
    )
    .reset_index()
)

tech_subtype_ts = utils.impute_empty_periods_all_ts(tech_subtype_ts, column)

utils.magnitude_and_growth(tech_subtype_ts, column, value)

,magnitude,growth,subtype
0,125003.289958,73.901417,AI
0,56932.294828,-17.969141,Immersive tech
0,578525.744551,41.941131,Internet
0,275438.436787,13.107150,Mobile
0,87175.512557,8.858934,Operations


In [242]:
fig = pu.ts_smooth(
    tech_subtype_ts,
    ["AI", "Immersive tech", "Internet", "Mobile", "Operations"],
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

## Insight 2: Applications

- Where are these technologies applied the most?
- Where do we see growth vs stagnation when it comes to applications?

In [96]:
tech_ids = (
    data_exploded_df
    # .query("type == 'Technology'")
    .query("subtype in @tech_subtypes")
    .query("year >= 2013")
    .drop_duplicates('id')
    .id.to_list()
)

tech_ids_5y = (
    data_exploded_df
    # .query("type == 'Technology'")
    .query("subtype in @tech_subtypes")
    .query("year >= 2019")
    .drop_duplicates('id')
    .id.to_list()
)

### Application distribution

In [97]:
column = 'type'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology' and type != 'General'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)


In [98]:
tech_applications_df

,type,counts,counts_prop,amount,amount_prop
0,Biosciences,9,0.018,9026.586836,0.002
1,Child care & preschool,102,0.199,463498.943387,0.11
2,Development & learning,147,0.287,1459553.275091,0.345
3,General,162,0.316,1441560.496722,0.341
4,Health,144,0.281,620837.404541,0.147
5,Parenting,105,0.205,273485.631675,0.065
6,Society,50,0.097,165733.006945,0.039
7,Technology,513,1.0,4232509.547608,1.0


In [99]:
fig = pu.ts_smooth(
    tech_applications_ts.query("year >= 2017"),
    tech_applications_ts[column].unique(),
    variable= "amount",
    variable_title = "Investment (£ thousands)",
    category_column = column,
    width = 400,
    height = 200,
    legend_orient='right',
)
pu.configure_plots(fig) 

alt.Chart(...)

In [100]:
utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount')

,magnitude,growth,type,counts
7,1805.317367,1748.296224,Biosciences,9
2,288312.099344,177.247812,General,162
1,291910.655018,44.765309,Development & learning,147
6,846501.909522,41.195625,Technology,513
3,124167.480908,40.466729,Health,144
5,33146.601389,22.534738,Society,50
0,92699.788677,-0.863038,Child care & preschool,102
4,54697.126335,-7.308249,Parenting,105


In [101]:
trends_df = utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount')
(
    tech_applications_df
    .merge(trends_df.drop('counts', axis=1), on='type')[['type', 'magnitude', 'growth', 'counts', 'counts_prop', 'amount', 'amount_prop']]
    .query("type != 'Technology' and type != 'General'")
)

,type,magnitude,growth,counts,counts_prop,amount,amount_prop
0,Biosciences,1805.317367,1748.296224,9,0.018,9026.586836,0.002
1,Child care & preschool,92699.788677,-0.863038,102,0.199,463498.943387,0.11
2,Development & learning,291910.655018,44.765309,147,0.287,1459553.275091,0.345
4,Health,124167.480908,40.466729,144,0.281,620837.404541,0.147
5,Parenting,54697.126335,-7.308249,105,0.205,273485.631675,0.065
6,Society,33146.601389,22.534738,50,0.097,165733.006945,0.039


In [102]:
from discovery_child_development.utils import chart_trends
importlib.reload(chart_trends);

chart_trends.estimate_trend_type(
    trends_df.query("type != 'Technology' and type != 'General'"), 
    magnitude_column='magnitude', 
    growth_column='growth'
)

,magnitude,growth,type,counts,trend_type_suggestion
7,1805.317367,1748.296224,Biosciences,9,emerging
1,291910.655018,44.765309,Development & learning,147,hot
3,124167.480908,40.466729,Health,144,hot
5,33146.601389,22.534738,Society,50,emerging
0,92699.788677,-0.863038,Child care & preschool,102,stable
4,54697.126335,-7.308249,Parenting,105,dormant


In [103]:
# scatter chart of trends_df
import altair as alt
alt.Chart(
    trends_df.query("type != 'Technology' and type != 'General' and type != 'Biosciences'")
).mark_point().encode(
    x='magnitude:Q',
    y='growth:Q',
    color='type:N',
    tooltip=['type', 'magnitude', 'growth']
)


alt.Chart(...)

In [531]:
# pd.set_option('display.max_colwidth', 200)
# (
#     data_exploded_df
#     .query('id in @tech_ids')
#     .query("type == 'Social'")
#     .drop_duplicates(['id'])
#     .sort_values('year', ascending=False)
# )

### Application distribution: More granular subtypes

In [1228]:
column = 'subtype'

tech_applications_df = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids_5y'), 
    column=column, 
    values=['id', 'amount']
)


tech_applications_ts = utils.get_data_distribution(
    data_exploded_df.query('id in @tech_ids').query("type != 'Technology'"),
    column=column, 
    values=['id', 'amount'],
    ts=True
)
tech_applications_df.query("type != 'Technology'").sort_values('amount', ascending=False)

,subtype,counts,counts_prop,amount,amount_prop,type
2,Cognitive development,49,0.096,866580.931841,0.205,Development & learning
10,Infancy,104,0.203,816533.215415,0.193,General
5,Games,64,0.125,628903.70653,0.149,General
13,Literacy,58,0.113,541413.478864,0.128,Development & learning
6,Health,132,0.257,514780.22023,0.122,Health
19,Operations,89,0.173,435877.562785,0.103,Child care & preschool
20,Parenting,105,0.205,273485.631675,0.065,Parenting
8,Inclusion,3,0.006,120438.550793,0.028,Society
27,Special educational needs,33,0.064,95654.311229,0.023,Development & learning
18,Nutrition & weight,8,0.016,78094.681684,0.018,Health


In [1229]:
(
    utils.get_data_magnitude_growth(data_exploded_df, ids=tech_ids, column=column, value='amount')
    .sort_values(['type', 'growth'], ascending=False)
)

,magnitude,growth,subtype,counts,type
9,125003.289958,73.901417,AI,173,Technology
13,578525.744551,41.941131,Internet,110,Technology
14,275438.436787,13.107150,Mobile,234,Technology
21,56932.294828,-17.969141,Immersive tech,62,Technology
2,24087.710159,1269.157450,Inclusion,3,Society
5,8125.582158,313.856279,Social services,3,Society
12,10043.062791,57.187056,Income,22,Society
16,3734.694462,8.561909,Labour market,12,Society
22,13797.732638,-41.600961,Community,20,Society
18,54697.126335,-7.308249,Parenting,105,Parenting


In [1230]:
cat_type = 'Development & learning'
cats = list(topics_df.query("type == @cat_type").subtype.unique())

In [1231]:
fig = pu.ts_smooth(
    tech_applications_ts.query("year >= 2017"),
    cats,
    variable= "amount",
    variable_title = "",
    category_column = column,
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

## Insight 3: Geographical insights


In [454]:
# data_exploded_df.dropna(subset=['amount']).head()

In [1073]:
data_countries_df = (
    data_exploded_df
    .dropna(subset=['country_code'])
    .query("type == 'Technology'")
    .query('subtype in @tech_subtypes')
    .drop_duplicates(['id'])    
)
country_codes = data_countries_df.country_code.unique()

growth_df = []
ts_counts = []
for country_code in country_codes:
    country_df = data_countries_df.query("country_code == @country_code")
    _ts_counts = utils.get_timeseries(country_df, column='amount').assign(country_code = country_code)
    growth_df.append(
        au.ts_magnitude_growth_(
            ts_df = _ts_counts,
            year_start = 2019,
            year_end = 2023  
        )
        .assign(country_code = country_code)
        .reset_index(drop=True)
    )
    ts_counts.append(_ts_counts)
growth_df = pd.concat(growth_df, ignore_index=True)
ts_counts = pd.concat(ts_counts, ignore_index=True)

In [1074]:
mag_growth_countries_df = (
    growth_df
    .sort_values('magnitude', ascending=False)
    .head(15)
)
mag_growth_countries_df

,magnitude,growth,country_code
3,523630.007604,147.721838,USA
11,158982.268980,13.455687,IND
1,46772.956806,-81.965750,CHN
8,37109.720306,-76.741053,GBR
4,11868.538098,602.395266,CAN
31,10437.474855,-23.922810,JPN
35,9495.637346,1626.850918,KOR
6,7872.595920,179.978612,ESP
5,7784.813000,135.579853,DEU
0,6010.114111,235.287954,FRA


In [1077]:
mag_growth_countries_df.query("country_code == 'USA'").magnitude.iloc[0]*5/1e+3

2618.1500380179605

In [1076]:
mag_growth_countries_df.query("country_code == 'GBR'").magnitude.iloc[0]*5/1e+3

185.5486015294242

In [756]:
print(94.34029633023444 / 333.0302899859875)
print(2107.9132502882894 / 5233.9416857739725)

0.2832784259179664
0.4027391547020227


In [443]:
countries = ['GBR', 'USA', 'IND', 'CHN']
fig = pu.ts_smooth(
    ts_counts.query("country_code in @countries"),
    countries,
    variable= "amount",
    variable_title = "",
    category_column = 'country_code',
    width = 300,
    height = 150,
    legend_orient='right',
)
pu.configure_plots(fig) 


alt.Chart(...)

## Detailed applications

In [299]:
importlib.reload(utils);
df = utils.get_counts_by_application(data_exploded_df, topics_df, 'amount', 'sum')
df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_crunchbase_amount.csv', index=False)

In [300]:
importlib.reload(utils);
df = utils.get_counts_by_application(data_exploded_df, topics_df)
df.to_csv(utils.PROJECT_DIR / 'outputs/data/tables/tech_applications_crunchbase_counts.csv', index=False)

## Check company data

In [113]:
# from discovery_child_development.getters import crunchbase
# cb_df = crunchbase.get_cb_from_s3('organizations')

In [18]:
cb_funding_df = pd.read_parquet(utils.INPUTS_DATA_DIR / "crunchbase/funding_rounds_full.parquet")

In [19]:
total_funding_gbp_df = cb_funding_df.drop_duplicates('funding_round_id').groupby("org_id").agg(total_funding_gbp = ('raised_amount_gbp', 'sum')).reset_index()

In [20]:
relevant_funding_gbp_df = data_df.groupby("org_id").agg(relevant_funding_gbp = ('amount', 'sum')).reset_index()

In [21]:
col_order = [
    "id",
    "name",
    "text",
    "country_code",
    "topics",
    'url',
    "cb_url",
    "relevant_funding_gbp",
    "total_funding_gbp",
    "num_funding_rounds",
    "last_funding_on",
]

In [27]:
companies_df = (
    utils.load_crunchbase_companies()
    .query("id not in @remove_ids")
    .assign(
        # ignore ids that are not in the corrected_ids dict
        topics=lambda x: x['id'].map(corrected_ids).fillna(x['topics'])
    )    
    .merge(cb_df[['id', 'homepage_url', 'country_code']].drop_duplicates(), on='id', how='left')
    .assign(text = lambda df: df.text.fillna(df.short_description)) 
    .merge(total_funding_gbp_df, left_on='id', right_on='org_id', how='left')
    .assign(total_funding_gbp = lambda df: df.total_funding_gbp.fillna(0))
    .assign(total_funding_gbp = lambda df: df.total_funding_gbp / 1000)
    .drop('org_id', axis=1)    
    .merge(relevant_funding_gbp_df, left_on='id', right_on='org_id', how='left')
    .assign(relevant_funding_gbp = lambda df: df.relevant_funding_gbp.fillna(0))
    .assign(relevant_funding_gbp = lambda df: df.relevant_funding_gbp / 1000)    
    .drop('org_id', axis=1)
    .rename(columns={'homepage_url': 'url'})
)[col_order]


In [29]:
col_order_exploded = [
    "id",
    "name",
    "text",
    "country_code",
    "url",
    "cb_url",
    "relevant_funding_gbp",
    "total_funding_gbp",
    "num_funding_rounds",
    "last_funding_on",
    "topic",
    "minor_category",
    "major_category",
    "topic_code",
]

In [44]:
companies_df_exploded = (
    companies_df
    .fillna({'topics': ''})
    .assign(topics = lambda df: df['topics'].apply(lambda x: [t.strip() for t in x.split(',') if isinstance(t, str)]))
    .explode('topics')
    .merge(topics_df, left_on='topics', right_on='topic', how='left', suffixes=('', '_topic'))
    .rename(columns={'topic': 'topic_code', 'type': 'major_category', 'subtype': 'minor_category', 'name_topic': 'topic'})
    .drop('topics', axis=1)
)[col_order_exploded]


In [45]:
companies_df_exploded.to_csv(PROJECT_DIR / "outputs/data/tables/crunchbase_final_exploded.csv", index=False)

In [46]:
companies_df_exploded.head(1)

,id,name,text,country_code,url,cb_url,relevant_funding_gbp,total_funding_gbp,num_funding_rounds,last_funding_on,topic,minor_category,major_category,topic_code
0,397845f1-40de-d4f2-8ce8-c0f44161a708,Toys R Us Iberia,Toys “R” Us is a toy and baby products retaile...,ESP,https://www.toysrus.es,https://www.crunchbase.com/organization/toys-r-us,0.0,2347.748486,2.0,2017-09-27,Games,Games,General,games


In [47]:
_companies_df = (
    companies_df_exploded
    .drop(
            [
                "name",
                "text",
                "country_code",
                "url",
                "cb_url",
                "relevant_funding_gbp",
                "total_funding_gbp",
                "num_funding_rounds",
                "last_funding_on",
            ], axis=1        
    )
    .groupby(
        [
                "id",
        ]
    )
    .agg(set)
    .reset_index()
    
)

In [266]:
len(data_df.drop_duplicates('id'))

2540

In [269]:
data_df.year.max()

2023

In [48]:
def check_if_nan_only_list_elements(x):
    return all([pd.isna(i) for i in x])

def convert_topic_columns(df):
    for col in ['topic', 'minor_category', 'major_category', 'topic_code']:
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: list(x) if check_if_nan_only_list_elements(x)==False else [])})
        df = df.assign(**{col : lambda df: df[col].apply(lambda x: ", ".join([xx for xx in x if isinstance(xx, str)]) if len(x)>0 else "")})
    return df

In [49]:
companies_df_final = (
    companies_df
    .drop("topics", axis=1)
    .merge(_companies_df, on='id')
    .rename(columns={'cb_url': 'url'})
    .pipe(convert_topic_columns)
)

In [50]:
companies_df_final.to_csv(PROJECT_DIR / "outputs/data/tables/crunchbase_final.csv", index=False)